약효분류군 - 성분코드 매핑 데이터

1. nhis 국민건강보험공단 데이터 : 급여항목만 제공
2. 건강보험심사평가원 : atc 매핑 테이블 = 식약처분류코드 (보건복지부 약효분류코드 아님)
https://docs.google.com/spreadsheets/d/19u-mKb0u6V_mPMFCc8yfGNEQJSTXyV1F2HyAB3aTFPc/edit?gid=1200370600#gid=1200370600
3. 건강보험심사평가원 : 매핑 관계모르면 조회불가
https://www.data.go.kr/data/15021027/openapi.do?recommendDataYn=Y
4. 빅데이터플랫폼 : 사용신청 어렵넨
https://hcdl.mohw.go.kr/static/data/dataApplyStep
5. 약제 급여 목록표 : got cha!!!! 분류번호 type = int  ***

https://www.hira.or.kr/bbsDummyKR.do?pgmid=HIRAA030014050000&brdScnBltNo=4&brdBltNo=1706&pageIndex=1&pageIndex2=1#none


In [6]:
import pandas as pd
import glob

filelist = glob.glob('../generic/MEDICINE_2025/*')

df = pd.read_parquet(filelist[0])
df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)
df


,STD_YYYYMM,WK_COMPN_4,AGE_G,SEX_TYPE,SIDO,YOYANG_CLSFC_CD_ADJ,OUT_IN_HOS,PRSCRPTN_TNDN_CNT,PRSCRPTN_AMT,PATIENT_CNT,PRSCRPTN_TOT_USE
0,202508,1007,0,1,11,1,OUT,2,4974,2,24.50
1,202508,1007,0,1,11,3,OUT,479,401155,347,2055.88
2,202508,1007,0,1,26,1,OUT,1,2110,1,10.00
3,202508,1007,0,1,26,3,OUT,47,40542,36,206.15
4,202508,1007,0,1,27,3,OUT,78,60923,54,333.74
...,...,...,...,...,...,...,...,...,...,...,...
575754,202508,8016,90,2,47,1,IN,7,644406,7,44.00
575755,202508,8016,90,2,48,1,IN,30,3680389,27,255.00
575756,202508,8016,90,2,50,1,IN,1,73560,1,5.00
575757,202508,8016,90,2,51,1,IN,19,2776524,18,188.00


In [1]:


import pandas as pd, glob, os

filelist = glob.glob('../data/*')


for f in filelist:
    if not f.lower().endswith(('.xlsx', '.xls')):
        continue
    try:
        xl = pd.ExcelFile(f)
        for s in xl.sheet_names:
            df = pd.read_excel(f, sheet_name=s, nrows=2)
            print(f"{os.path.basename(f)} [{s}] → {df.shape[1]}컬럼")
    except Exception as e:
        print(f, 'ERR', e)

약제급여목록_2025.6.1.xlsx [2025년6월1일_(21,983)] → 12컬럼
약제급여목록_2023.6.1.xlsx [2023년6월1일_(23,505)] → 13컬럼
약제급여목록_2024.6.1.xlsx [2024년6월1일_(22,861)] → 13컬럼


In [18]:
path = filelist[0]

def clean_data(path):
    df = pd.read_excel(path, dtype={'제품코드': str, '주성분코드': str})

    # 1) 제품행 판별
    is_product = df['제품코드'].str.fullmatch(r'\d+', na=False)

    # 2) 성분명 ffill
    df['성분명'] = df['제품코드'].where(~is_product).ffill()

    # 3) 제품행만 (성분명 컬럼은 자동 포함됨)
    df_clean = df[is_product].copy()

    # 4) 컬럼 순서만 재배치 (insert 대신 reindex)
    cols = ['연번', '투여', '분류', '주성분코드', '성분명',
            '제품코드', '제품명', '업체명', '규격', '단위', '상한금액', '전일', '비고']
    df_clean = df_clean[cols].reset_index(drop=True)

    # 5) 상한금액 숫자화
    df_clean['상한금액'] = (df_clean['상한금액'].astype(str)
                        .str.replace(',', '', regex=False)
                        .pipe(pd.to_numeric, errors='coerce'))

    print(df_clean.shape, df_clean.dtypes)
    display(df_clean.head())
    return df_clean


In [19]:
# df_clean[df_clean['분류']=='421']


# 1. 반복문 밖에서 빈 set(집합)을 먼저 만들어줍니다.
meft_generic_set = set()

for ipath in filelist:
    idf = clean_data(ipath)
    
    # 결측치(NaN) 방지를 위해 문자열로 변환 후 슬라이싱 (안전한 코딩)
    idf['WK_COMPN_4'] = idf['주성분코드'].astype(str).str[:4]
    
    # 2. set.update()를 사용하여 (분류, WK_COMPN_4) 튜플들을 한 번에 추가합니다.
    # zip()을 사용해 튜플 형태로 만들어주면 update가 알아서 중복을 걸러냅니다.
    temp_df = idf[['분류','WK_COMPN_4']].drop_duplicates()
    meft_generic_set.update(zip(temp_df['분류'], temp_df['WK_COMPN_4']))
    
    # (참고) 기존에 작성하셨던 리스트 append 방식은 이제 필요 없습니다!
    # meft_generic_map.append(...) 

# 3. 반복문이 끝난 후, 최종 결과를 데이터프레임으로 변환합니다.
meft_generic_map_df = pd.DataFrame(list(meft_generic_set), columns=['분류', 'WK_COMPN_4'])

# 결과 확인
print(meft_generic_map_df.head())
print(f"총 관계 수: {len(meft_generic_map_df)}")

(21983, 13) 연번         int64
투여           str
분류         int64
주성분코드        str
성분명          str
제품코드         str
제품명          str
업체명          str
규격           str
단위           str
상한금액     float64
전일           str
비고           str
dtype: object


,연번,투여,분류,주성분코드,성분명,제품코드,제품명,업체명,규격,단위,상한금액,전일,비고
0,2,내복,112,130830ASY,chloral hydrate 9.5g(0.1g/mL),645302132,포크랄시럽(포수클로랄)_(9.5g/95mL),한림제약(주),95(1),mL/병,129.0,전문,NaN
1,4,내복,112,130832ASY,chloral hydrate 1g(0.1g/mL),645302133,포크랄시럽(포수클로랄)_(1g/10mL),한림제약(주),10,mL/병,1260.0,전문,NaN
2,6,내복,112,130833ASY,chloral hydrate 0.5g(0.1g/mL),645302135,포크랄시럽(포수클로랄)_(0.5g/5mL),한림제약(주),5,mL/병,834.0,전문,NaN
3,8,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),651904420,명세핀정3밀리그램(독세핀염산염)_(3.39mg/1정),명인제약(주),1,정,96.0,전문,NaN
4,9,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),640006660,사일레노정3밀리그램(독세핀염산염)_(3.39mg/1정),에이치케이이노엔(주),1,정,97.0,전문,NaN


(23505, 13) 연번         int64
투여           str
분류         int64
주성분코드        str
성분명          str
제품코드         str
제품명          str
업체명          str
규격        object
단위           str
상한금액     float64
전일           str
비고           str
dtype: object


,연번,투여,분류,주성분코드,성분명,제품코드,제품명,업체명,규격,단위,상한금액,전일,비고
0,2,내복,112,130830ASY,chloral hydrate 9.5g(0.1g/mL),645302132,포크랄시럽(포수클로랄)_(9.5g/95mL),한림제약(주),95(1),mL/병,129.0,전문,NaN
1,4,내복,112,130832ASY,chloral hydrate 1g(0.1g/mL),645302133,포크랄시럽(포수클로랄)_(1g/10mL),한림제약(주),10,mL/병,1260.0,전문,NaN
2,6,내복,112,130833ASY,chloral hydrate 0.5g(0.1g/mL),645302135,포크랄시럽(포수클로랄)_(0.5g/5mL),한림제약(주),5,mL/병,847.0,전문,NaN
3,8,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),651904420,명세핀정3밀리그램(독세핀염산염)_(3.39mg/1정),명인제약(주),1,정,96.0,전문,NaN
4,9,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),640006660,사일레노정3밀리그램(독세핀염산염)_(3.39mg/1정),에이치케이이노엔(주),1,정,97.0,전문,NaN


(22992, 13) 연번         int64
투여           str
분류         int64
주성분코드        str
성분명          str
제품코드         str
제품명          str
업체명          str
규격        object
단위           str
상한금액     float64
전일           str
비고           str
dtype: object


,연번,투여,분류,주성분코드,성분명,제품코드,제품명,업체명,규격,단위,상한금액,전일,비고
0,2,내복,112,130830ASY,chloral hydrate 9.5g(0.1g/mL),645302132,포크랄시럽(포수클로랄)_(9.5g/95mL),한림제약(주),95(1),mL/병,129.0,전문,NaN
1,4,내복,112,130832ASY,chloral hydrate 1g(0.1g/mL),645302133,포크랄시럽(포수클로랄)_(1g/10mL),한림제약(주),10,mL/병,1260.0,전문,NaN
2,6,내복,112,130833ASY,chloral hydrate 0.5g(0.1g/mL),645302135,포크랄시럽(포수클로랄)_(0.5g/5mL),한림제약(주),5,mL/병,847.0,전문,"24.7.1., 834원"
3,8,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),651904420,명세핀정3밀리그램(독세핀염산염)_(3.39mg/1정),명인제약(주),1,정,96.0,전문,NaN
4,9,내복,112,149203ATB,doxepin hydrochloride (as doxepin 3mg),640006660,사일레노정3밀리그램(독세핀염산염)_(3.39mg/1정),에이치케이이노엔(주),1,정,97.0,전문,NaN


    분류 WK_COMPN_4
0  124       2579
1  121       1838
2  214       6322
3  617       1730
4  142       4514
총 관계 수: 2470


In [22]:
# meft_generic_set
print(meft_generic_map_df.shape)
meft_generic_map_df[meft_generic_map_df['분류']==112]

(2470, 2)


,분류,WK_COMPN_4
18,112,1308
223,112,1618
267,112,1606
322,112,2117
399,112,6804
1520,112,7174
1524,112,2435
1627,112,1952
1827,112,2505
1855,112,2106


In [23]:
meft_generic_map_df.to_csv('../data/meft_cmpn4_map.csv',index=False)